Workflows & Job Orchestration

In [0]:
dbutils.widgets.text("source_path", "")
dbutils.widgets.dropdown("layer", "bronze", ["bronze", "silver", "gold"])


In [0]:
source_path = dbutils.widgets.get("source_path")
layer = dbutils.widgets.get("layer")

print("Running layer:", layer)
print("Source path:", source_path)


Running layer: bronze
Source path: 


In [0]:
from pyspark.sql import functions as F

source_path = dbutils.widgets.get("source_path")
bronze_path = "/Volumes/workspace/ecommerce/bronze/events"

if not source_path:
    print("Error: source_path widget is empty. Please provide a valid path to the source CSV file.")
else:
    raw = spark.read.csv(
        source_path,
        header=True,
        inferSchema=True
    )
    bronze = raw.withColumn("ingestion_ts", F.current_timestamp())
    bronze.write.format("delta") \
        .mode("overwrite") \
        .save(bronze_path)
    print("BRONZE completed")


Error: source_path widget is empty. Please provide a valid path to the source CSV file.


In [0]:
from pyspark.sql import functions as F

bronze_path = "/Volumes/workspace/ecommerce/bronze/events"
silver_path = "/Volumes/workspace/ecommerce/silver/events"

bronze = spark.read.format("delta").load(bronze_path)

silver = (
    bronze
    .filter(F.col("price").isNotNull())
    .filter(F.col("price") > 0)
    .filter(F.col("price") < 10000)
    .dropDuplicates(["user_session", "event_time"])
    .withColumn("event_date", F.to_date("event_time"))
)

silver.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("SILVER completed")


SILVER completed


In [0]:
from pyspark.sql import functions as F

silver_path = "/Volumes/workspace/ecommerce/silver/events"
gold_path = "/Volumes/workspace/ecommerce/gold/product_performance"

silver = spark.read.format("delta").load(silver_path)

gold = (
    silver
    .groupBy("product_id")
    .agg(
        F.count(F.when(F.col("event_type") == "view", True)).alias("views"),
        F.count(F.when(F.col("event_type") == "purchase", True)).alias("purchases"),
        F.round(
            F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))), 2
        ).alias("revenue")
    )
    .withColumn(
        "conversion_rate",
        F.round(F.try_divide(F.col("purchases"), F.col("views")) * 100, 2)
    )
)

gold.write.format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("GOLD completed")


GOLD completed
